# 🚀 RAG Embedding Model Migration: Dual-Index Evaluation & CI-Gates

Dieses Notebook demonstriert ein strukturiertes **Zero-Downtime-Migrationsverfahren** für Embedding-Modelle in RAG-Systemen:
1. **Baseline-Index (v1):** `intfloat/multilingual-e5-base` (Dimension: 768)
2. **Kandidaten-Index (v2):** `BAAI/bge-m3` (Dimension: 1024)
3. **Fehlerhafter Reindex (v2-bad):** Simuliert ein vergessenes Dokument (Silent Corpus Drop).

**Evaluierte Metriken:** Recall@5, MRR@5, nDCG@5 sowie die **Critical Query Pass Rate**.

## Abhängigkeiten installieren & Importe

In [1]:
from IPython.core.display import Math
# Paketinstallation (leise)
%pip install -q sentence-transformers numpy pandas tabulate

import json
import math
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple, Set, Any

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Ausgabeformatierung für DataFrames im Notebook
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", None)
print("Imports und Umgebung erfolgreich eingerichtet.")

Imports und Umgebung erfolgreich eingerichtet.


## Datensatz definieren (Korpus & Golden Queries)

In [2]:
# 1. Korpus: Dokumentations-Chunks
DOCUMENTS = [
    {"id": "docs/configuration.md#http-timeout", "title": "HTTP-Timeout konfigurieren", "text": "Die Umgebungsvariable IMPORT_HTTP_TIMEOUT bestimmt das maximale HTTP-Timeout für Datenimporte in Sekunden. Der Standardwert beträgt 30. Beispiel: IMPORT_HTTP_TIMEOUT=120 erhöht das Timeout auf zwei Minuten.", "version": "2.4", "category": "configuration"},
    {"id": "docs/configuration.md#retry-policy", "title": "Retry-Verhalten konfigurieren", "text": "IMPORT_RETRY_ATTEMPTS legt fest, wie oft ein fehlgeschlagener Import erneut versucht wird. Der Standardwert ist 3. Zwischen Versuchen verwendet der Importer exponentielles Backoff.", "version": "2.4", "category": "configuration"},
    {"id": "docs/security.md#oauth-refresh", "title": "OAuth Refresh Tokens", "text": "OAuth Refresh Tokens werden verschlüsselt gespeichert. Sie dürfen nicht in Logs, Fehlermeldungen oder Browser-Local-Storage erscheinen. Beim Widerruf eines Tokens werden aktive Sitzungen beim nächsten Refresh beendet.", "version": "2.4", "category": "security"},
    {"id": "docs/security.md#api-keys", "title": "API-Schlüssel rotieren", "text": "API-Schlüssel können im Adminbereich rotiert werden. Nach der Rotation bleibt der vorherige Schlüssel 24 Stunden gültig, damit Deployments schrittweise aktualisiert werden können.", "version": "2.4", "category": "security"},
    {"id": "docs/release-notes.md#v2-4-postgres", "title": "PostgreSQL-Unterstützung in Version 2.4", "text": "Version 2.4 unterstützt PostgreSQL 15, 16 und 17. PostgreSQL 14 wird weiterhin gelesen, erhält aber keine neuen migrationsspezifischen Tests.", "version": "2.4", "category": "release-notes"},
    {"id": "docs/release-notes.md#v2-3-postgres", "title": "PostgreSQL-Unterstützung in Version 2.3", "text": "Version 2.3 unterstützt PostgreSQL 14, 15 und 16. PostgreSQL 17 wird von dieser Version nicht unterstützt.", "version": "2.3", "category": "release-notes"},
    {"id": "docs/troubleshooting.md#network-import", "title": "Import scheitert mit Netzwerkfehler", "text": "Bei wiederholten Netzwerkfehlern während eines Imports sollten zuerst DNS-Auflösung, Firewall-Regeln und der Zielendpunkt geprüft werden. Danach können IMPORT_HTTP_TIMEOUT und IMPORT_RETRY_ATTEMPTS erhöht werden.", "version": "2.4", "category": "troubleshooting"},
    {"id": "docs/cli.md#dry-run", "title": "Migration als Dry Run ausführen", "text": "Der Befehl app migrate --dry-run prüft Migrationen, ohne Datenbankänderungen auszuführen. Er zeigt die geplanten Schritte und beendet sich bei nicht erfüllten Voraussetzungen mit einem Fehlercode.", "version": "2.4", "category": "cli"},
    {"id": "docs/cli.md#rollback", "title": "Migration zurückrollen", "text": "Der Befehl app migrate rollback setzt die zuletzt erfolgreiche Datenbankmigration zurück. Vor einem Rollback sollte ein Datenbank-Snapshot erstellt und die Anwendung in den Wartungsmodus versetzt werden.", "version": "2.4", "category": "cli"},
    {"id": "docs/architecture.md#background-workers", "title": "Hintergrund-Worker", "text": "Importe laufen in Hintergrund-Workern. Die Anzahl paralleler Worker wird mit IMPORT_WORKERS konfiguriert. Ein höherer Wert kann den Durchsatz steigern, erhöht aber die Last auf Datenbank und Zielsystem.", "version": "2.4", "category": "architecture"},
    {"id": "docs/access-control.md#project-roles", "title": "Projektrollen", "text": "Die Rollen Viewer, Editor und Admin steuern den Zugriff auf Projekte. Viewer dürfen Inhalte lesen, Editor dürfen Inhalte ändern und Admin dürfen Mitglieder sowie Projekteinstellungen verwalten.", "version": "2.4", "category": "security"},
    {"id": "docs/observability.md#structured-logs", "title": "Strukturierte Logs", "text": "Setzen Sie LOG_FORMAT=json, um strukturierte JSON-Logs zu erzeugen. Jede Anfrage enthält request_id, route, status_code und duration_ms. Secrets wie Tokens und API-Schlüssel werden maskiert.", "version": "2.4", "category": "observability"},
]

# 2. Golden Evaluation Queries
GOLDEN_QUERIES = [
    {"id": "timeout-001", "query": "Wie erhöhe ich das Timeout für einen Datenimport?", "relevant_chunk_ids": ["docs/configuration.md#http-timeout"], "segment": "de-config", "critical": True},
    {"id": "retry-001", "query": "Wie viele Importversuche gibt es standardmäßig und wie ändere ich sie?", "relevant_chunk_ids": ["docs/configuration.md#retry-policy"], "segment": "de-config", "critical": False},
    {"id": "oauth-001", "query": "Wo dürfen OAuth Refresh Tokens auf keinen Fall gespeichert werden?", "relevant_chunk_ids": ["docs/security.md#oauth-refresh"], "segment": "de-security", "critical": True},
    {"id": "key-rotation-001", "query": "Wie lange funktioniert ein alter API-Schlüssel nach einer Rotation noch?", "relevant_chunk_ids": ["docs/security.md#api-keys"], "segment": "de-security", "critical": True},
    {"id": "postgres-001", "query": "Ist PostgreSQL 17 mit Version 2.4 kompatibel?", "relevant_chunk_ids": ["docs/release-notes.md#v2-4-postgres"], "segment": "de-version", "critical": True},
    {"id": "postgres-002", "query": "Wird PostgreSQL 17 von Version 2.3 unterstützt?", "relevant_chunk_ids": ["docs/release-notes.md#v2-3-postgres"], "segment": "de-version", "critical": True},
    {"id": "network-001", "query": "Was soll ich bei wiederholten Netzwerkfehlern eines Imports prüfen?", "relevant_chunk_ids": ["docs/troubleshooting.md#network-import"], "segment": "de-troubleshooting", "critical": False},
    {"id": "dry-run-001", "query": "Wie kann ich eine Migration prüfen, ohne Daten zu verändern?", "relevant_chunk_ids": ["docs/cli.md#dry-run"], "segment": "de-cli", "critical": True},
    {"id": "rollback-001", "query": "Wie rolle ich die letzte erfolgreiche Datenbankmigration zurück?", "relevant_chunk_ids": ["docs/cli.md#rollback"], "segment": "de-cli", "critical": True},
    {"id": "workers-001", "query": "Welche Einstellung steuert die Anzahl paralleler Importprozesse?", "relevant_chunk_ids": ["docs/architecture.md#background-workers"], "segment": "de-architecture", "critical": False},
    {"id": "roles-001", "query": "Welche Berechtigung wird benötigt, um Projekteinstellungen zu verwalten?", "relevant_chunk_ids": ["docs/access-control.md#project-roles"], "segment": "de-security", "critical": False},
    {"id": "logs-001", "query": "Wie aktiviere ich JSON-Logs und welche Felder enthalten sie?", "relevant_chunk_ids": ["docs/observability.md#structured-logs"], "segment": "de-observability", "critical": False},
]

print(f"Geladen: {len(DOCUMENTS)} Dokumente und {len(GOLDEN_QUERIES)} Test-Queries.")

Geladen: 12 Dokumente und 12 Test-Queries.


## Core-Engine (Metriken, Index & Embeddings)

In [3]:
# --- 1. Evaluierungs-Metriken ---
def reciprocal_rank(retrieved_ids: List[str], relevant_ids: Set[str]) -> float:
  for rank, chunk_id in enumerate(retrieved_ids, start=1):
    if chunk_id in relevant_ids:
      return 1.0 / rank
  return 0.0

def recall_at_k(retrieved_ids: List[str], relevant_ids: Set[str], k: int) -> float:
  return float(bool(set(retrieved_ids[:k]) & relevant_ids))

def dcg_at_k(retrieved_ids: List[str], relevant_ids: Set[str], k: int) -> float:
  score = 0.0
  for rank, chunk_id in enumerate(retrieved_ids[:k], start=1):
    if chunk_id in relevant_ids:
      score += 1.0 / math.log2(rank + 1)
  return score

def ndcg_at_k(retrieved_ids: List[str], relevant_ids: Set[str], k: int) -> float:
  ideal_hits = min(len(relevant_ids), k)
  ideal = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
  return dcg_at_k(retrieved_ids, relevant_ids, k) / ideal if ideal else 0.0

# --- 2. In-Memory Vektor-Index ---
class LocalVectorIndex:
  def __init__(self, records: List[Dict], vectors: np.ndarray):
    self.records = records
    self.vectors = vectors.astype(np.float32)

  def search(self, query_vector: np.ndarray, limit: int = 5) -> List[Dict]:
    scores = self.vectors @ query_vector.astype(np.float32)
    positions = np.argsort(-scores)[:limit]
    return [{**self.records[idx], "score": float(scores[idx])} for idx in positions]

# --- 3. Embedding Pipeline (inkl. Prefix-Handling für E5) ---
E5_MODEL = "intfloat/multilingual-e5-base"

def encode_passages(model: SentenceTransformer, model_name: str, texts: List[str]) -> np.ndarray:
  if model_name == E5_MODEL:
    texts = [f"passage: {t}" for t in texts]
  return model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

def encode_queries(model: SentenceTransformer, model_name: str, queries: List[str]) -> np.ndarray:
  if model_name == E5_MODEL:
    queries = [f"query: {q}" for q in queries]
  return model.encode(queries, normalize_embeddings=True, show_progress_bar=False)

## Indizes aufbauen & Evaluierungs-Harness

In [4]:
MODELS = {
    "v1": "intfloat/multilingual-e5-base",
    "v2": "BAAI/bge-m3",
    "v2-bad": "BAAI/bge-m3",
}

# Simulierter Reindex-Fehler: v2-bad fehlt der PostgreSQL-17-Chunk
EXCLUDED_CHUNKS = {
    "v2-bad": {"docs/release-notes.md#v2-4-postgres"}
}

# Caches verhindern doppeltes Laden von Modellen & bewahren die Indizes für den Router auf
MODEL_CACHE: Dict[str, SentenceTransformer] = {}
INDEX_CACHE: Dict[str, LocalVectorIndex] = {}

def run_evaluation(version: str, top_k: int = 5) -> Dict[str, Any]:
    model_name = MODELS[version]
    print(f"\nBerechne Index & evaluiere '{version}' ({model_name})...")

    # 1. Korpus filtern
    excluded = EXCLUDED_CHUNKS.get(version, set())
    docs = [d for d in DOCUMENTS if d["id"] not in excluded]

    # 2. Modell aus Cache holen oder einmalig laden
    if model_name not in MODEL_CACHE:
        print(f"  -> Lade Modell '{model_name}' erstmalig in den Speicher...")
        MODEL_CACHE[model_name] = SentenceTransformer(model_name)
    else:
        print(f"  -> Nutze gecachtes Modell '{model_name}'.")
    model = MODEL_CACHE[model_name]

    # 3. Vektoren berechnen & Index cachen
    doc_vectors = encode_passages(model, model_name, [d["text"] for d in docs])
    index = LocalVectorIndex(docs, doc_vectors)
    INDEX_CACHE[version] = index

    # 4. Queries abfragen
    query_texts = [q["query"] for q in GOLDEN_QUERIES]
    query_vectors = encode_queries(model, model_name, query_texts)

    query_results = []
    for item, q_vec in zip(GOLDEN_QUERIES, query_vectors):
        hits = index.search(q_vec, limit=top_k)
        retrieved_ids = [h["id"] for h in hits]
        relevant_ids = set(item["relevant_chunk_ids"])

        query_results.append({
            "id": item["id"],
            "query": item["query"],
            "segment": item["segment"],
            "critical": item["critical"],
            "relevant_chunk_ids": item["relevant_chunk_ids"],
            "retrieved_chunk_ids": retrieved_ids,
            "hits": [{"id": h["id"], "title": h["title"], "score": h["score"]} for h in hits],
            "recall_at_k": recall_at_k(retrieved_ids, relevant_ids, top_k),
            "rr": reciprocal_rank(retrieved_ids, relevant_ids),
            "ndcg_at_k": ndcg_at_k(retrieved_ids, relevant_ids, top_k),
        })

    critical_rows = [r for r in query_results if r["critical"]]

    metrics = {
        f"recall_at_{top_k}": sum(r["recall_at_k"] for r in query_results) / len(query_results),
        f"mrr_at_{top_k}": sum(r["rr"] for r in query_results) / len(query_results),
        f"ndcg_at_{top_k}": sum(r["ndcg_at_k"] for r in query_results) / len(query_results),
        "critical_pass_rate": (
            sum(r["recall_at_k"] for r in critical_rows) / len(critical_rows)
            if critical_rows else 1.0
        ),
    }

    return {
        "version": version,
        "model_name": model_name,
        "doc_count": len(docs),
        "metrics": metrics,
        "query_results": query_results,
    }

# Führe alle drei Szenarien aus
report_v1 = run_evaluation("v1")
report_v2 = run_evaluation("v2")
report_v2_bad = run_evaluation("v2-bad")
print("\nAlle Evaluationen abgeschlossen.")


Berechne Index & evaluiere 'v1' (intfloat/multilingual-e5-base)...
  -> Lade Modell 'intfloat/multilingual-e5-base' erstmalig in den Speicher...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (22 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]


Berechne Index & evaluiere 'v2-bad' (BAAI/bge-m3)...
  -> Nutze gecachtes Modell 'BAAI/bge-m3'.

Alle Evaluationen abgeschlossen.


## Management Dashboard (Vergleichs-Tabelle)

In [5]:
# Erzeuge Vergleichsübersicht
metrics_keys = ["recall_at_5", "mrr_at_5", "ndcg_at_5", "critical_pass_rate"]

summary_data = []
for m in metrics_keys:
  val_v1 = report_v1["metrics"][m]
  val_v2 = report_v2["metrics"][m]
  val_v2_bad = report_v2_bad["metrics"][m]
  delta_v2 = val_v2 - val_v1

  summary_data.append({
      "Metric": m,
      "Baseline (v1)": f"{val_v1:.3f}",
      "Candidate (v2)": f"{val_v2:.3f}",
      "Delta (v2 vs v1)": f"{delta_v2:+.3f}",
      "Incomplete Reindex (v2-bad)": f"{val_v2_bad:.3f}"
  })

df_summary = pd.DataFrame(summary_data)
print("\n MiGRATION DECISION MATRIX")
display(df_summary)


 MiGRATION DECISION MATRIX


,Metric,Baseline (v1),Candidate (v2),Delta (v2 vs v1),Incomplete Reindex (v2-bad)
0,recall_at_5,1.000,1.000,+0.000,0.917
1,mrr_at_5,0.958,0.958,+0.000,0.917
2,ndcg_at_5,0.969,0.969,+0.000,0.917
3,critical_pass_rate,1.000,1.000,+0.000,0.857


## Regression Analysis (Kritische Queries im Detail)

In [6]:
v1_map = {r["id"]: r for r in report_v1["query_results"]}
bad_map = {r["id"]: r for r in report_v2_bad["query_results"]}

regressions = []
for q_id, bad_row in bad_map.items():
  v1_row = v1_map[q_id]
  if bad_row["recall_at_k"] < v1_row["recall_at_k"]:
    regressions.append({
        "Query ID": q_id,
        "Query": bad_row["query"],
        "Critical": "🚨 JA" if bad_row["critical"] else "NEIN",
        "Expected": bad_row["relevant_chunk_ids"][0],
        "Top-1 Retrieved": bad_row["retrieved_chunk_ids"][0],
        "Recall Baseline": v1_row["recall_at_k"],
        "Recall Bad": bad_row["recall_at_k"]
    })

print("\n⚠️ REGRESSIONSANALYSE (Simulierter fehlerhafter Reindex v2-bad):")
df_regressions = pd.DataFrame(regressions)
display(df_regressions)


⚠️ REGRESSIONSANALYSE (Simulierter fehlerhafter Reindex v2-bad):


,Query ID,Query,Critical,Expected,Top-1 Retrieved,Recall Baseline,Recall Bad
0,postgres-001,Ist PostgreSQL 17 mit Version 2.4 kompatibel?,🚨 JA,docs/release-notes.md#v2-4-postgres,docs/release-notes.md#v2-3-postgres,1.0,0.0


## CI/CD Quality Gates (Automatisierte Freigabe-Tests)

In [7]:
# Emulation der GitHub-Actions CI-Prüfungen
MAX_RECALL_DROP = 0.02
MAX_MRR_DROP = 0.02

def test_migration_release_gate(baseline: dict, candidate: dict):
    b_m = baseline["metrics"]
    c_m = candidate["metrics"]

    print(f"🚦 Prüfe Release-Gate für Candidate: {candidate['version']} ({candidate['model_name']})")

    # Check 1: Recall Drop
    recall_drop = b_m["recall_at_5"] - c_m["recall_at_5"]
    assert recall_drop <= MAX_RECALL_DROP, (
        f"❌ FAILED: Recall@5 Drop überschreitet Schwelle! ({b_m['recall_at_5']:.3f} -> {c_m['recall_at_5']:.3f})"
    )
    print(f"  ✅ Check 1: Recall@5 im Toleranzbereich (Delta: {-recall_drop:+.3f}).")

    # Check 2: MRR Drop
    mrr_drop = b_m["mrr_at_5"] - c_m["mrr_at_5"]
    assert mrr_drop <= MAX_MRR_DROP, (
        f"❌ FAILED: MRR@5 Drop überschreitet Schwelle! ({b_m['mrr_at_5']:.3f} -> {c_m['mrr_at_5']:.3f})"
    )
    print(f"  ✅ Check 2: MRR@5 im Toleranzbereich (Delta: {-mrr_drop:+.3f}).")

    # Check 3: Critical Queries Pass Rate
    failed_critical = [
        row["id"] for row in candidate["query_results"]
        if row["critical"] and row["recall_at_k"] == 0
    ]
    assert not failed_critical, f"❌ FAILED: Kritische Queries ohne Treffer: {failed_critical}"
    print("  ✅ Check 3: 100% Pass-Rate bei geschäftskritischen Fragen.")

    print(f"🎉 FREIGABE ERTEILT: Index '{candidate['version']}' ist produktionsbereit!\n")

# 1. Gate für valides Modell v2 prüfen:
test_migration_release_gate(report_v1, report_v2)

# 2. Gate für fehlerhaftes Modell v2-bad prüfen (sollte Exception werfen):
print("Simuliere Gate-Prüfung für unvollständigen Index (v2-bad):")
try:
    test_migration_release_gate(report_v1, report_v2_bad)
except AssertionError as e:
    print(e)

🚦 Prüfe Release-Gate für Candidate: v2 (BAAI/bge-m3)
  ✅ Check 1: Recall@5 im Toleranzbereich (Delta: -0.000).
  ✅ Check 2: MRR@5 im Toleranzbereich (Delta: -0.000).
  ✅ Check 3: 100% Pass-Rate bei geschäftskritischen Fragen.
🎉 FREIGABE ERTEILT: Index 'v2' ist produktionsbereit!

Simuliere Gate-Prüfung für unvollständigen Index (v2-bad):
🚦 Prüfe Release-Gate für Candidate: v2-bad (BAAI/bge-m3)
❌ FAILED: Recall@5 Drop überschreitet Schwelle! (1.000 -> 0.917)


In [8]:
def analyze_margins(report: dict) -> dict:
    top1_scores = []
    margins = []

    for row in report["query_results"]:
        hits = row["hits"]
        if len(hits) >= 2:
            top1_scores.append(hits[0]["score"])
            margins.append(hits[0]["score"] - hits[1]["score"])

    return {
        "Index": report["version"],
        "Modell": report["model_name"],
        "Dimension": 1024 if "bge-m3" in report["model_name"] else 768,
        "Dokumente": report["doc_count"],
        "Ø Top-1 Cosine": sum(top1_scores) / len(top1_scores),
        "Ø Margin (Top1 - Top2)": sum(margins) / len(margins),
    }

df_specs = pd.DataFrame([
    analyze_margins(report_v1),
    analyze_margins(report_v2),
    analyze_margins(report_v2_bad),
])

print("📐 MODELL-SPEZIFIKATIONEN & TRENNSCHÄRFE:")
display(df_specs)

# Detailblick auf den Silent Near-Miss in v2-bad
near_miss = next(r for r in report_v2_bad["query_results"] if r["id"] == "postgres-001")
print(f"\n🔎 SILENT NEAR-MISS DETAIL ('{near_miss['query']}' in v2-bad):")
for rank, hit in enumerate(near_miss["hits"][:3], start=1):
    print(f"  Rang {rank}: {hit['id']} | Cosine-Score: {hit['score']:.4f} | '{hit['title']}'")

📐 MODELL-SPEZIFIKATIONEN & TRENNSCHÄRFE:


,Index,Modell,Dimension,Dokumente,Ø Top-1 Cosine,Ø Margin (Top1 - Top2)
0,v1,intfloat/multilingual-e5-base,768,12,0.893,0.055
1,v2,BAAI/bge-m3,1024,12,0.705,0.138
2,v2-bad,BAAI/bge-m3,1024,11,0.705,0.187



🔎 SILENT NEAR-MISS DETAIL ('Ist PostgreSQL 17 mit Version 2.4 kompatibel?' in v2-bad):
  Rang 1: docs/release-notes.md#v2-3-postgres | Cosine-Score: 0.7312 | 'PostgreSQL-Unterstützung in Version 2.3'
  Rang 2: docs/security.md#api-keys | Cosine-Score: 0.4048 | 'API-Schlüssel rotieren'
  Rang 3: docs/architecture.md#background-workers | Cosine-Score: 0.3696 | 'Hintergrund-Worker'


## Dual-Index Router mit Fallback

In [9]:
class DualIndexRouter:
    """
    Simuliert einen produktiven Router für Zero-Downtime-Migrationen.
    Kapselt jeweils Ziel-Index und zugehöriges Embedding-Modell und
    ermöglicht unterbrechungsfreie Cutover- und Rollback-Operationen.
    """
    def __init__(self, index_v1, model_v1, model_name_v1: str,
                 index_v2, model_v2, model_name_v2: str,
                 active_version: str = "v1"):
        self.routes = {
            "v1": {"index": index_v1, "model": model_v1, "model_name": model_name_v1},
            "v2": {"index": index_v2, "model": model_v2, "model_name": model_name_v2},
        }
        self.active_version = active_version

    def query(self, query_text: str, top_k: int = 5) -> dict:
        route = self.routes[self.active_version]
        query_vector = encode_queries(route["model"], route["model_name"], [query_text])[0]
        results = route["index"].search(query_vector, limit=top_k)

        return {
            "served_by": self.active_version,
            "top_hit": results[0]["id"] if results else None,
            "score": round(results[0]["score"], 4) if results else 0.0,
        }

    def promote_candidate(self, candidate_passed_gates: bool):
        if candidate_passed_gates:
            print("🟢 Gate bestanden: Schalte Produktions-Traffic unterbrechungsfrei auf v2 um!")
            self.active_version = "v2"
        else:
            print("🔴 Gate NICHT bestanden: Traffic bleibt sicher auf v1 (Rollback/Hold)!")

    def rollback(self):
        print("⏪ Notfall-Rollback ausgeführt: Traffic läuft sofort wieder über v1!")
        self.active_version = "v1"

# Router mit echten Indizes & Modellen aus den Caches instanziieren
router = DualIndexRouter(
    index_v1=INDEX_CACHE["v1"],
    model_v1=MODEL_CACHE[MODELS["v1"]],
    model_name_v1=MODELS["v1"],
    index_v2=INDEX_CACHE["v2"],
    model_v2=MODEL_CACHE[MODELS["v2"]],
    model_name_v2=MODELS["v2"],
    active_version="v1",
)

test_query = "Ist PostgreSQL 17 mit Version 2.4 kompatibel?"

print(f"Status vor CI-Gate: Aktiv = {router.active_version}")
print("Antwort v1:", router.query(test_query))

# Simulierter Freigabe-Entscheid (mit realem Gate-Ergebnis von v2):
gate_passed = (
    report_v2["metrics"]["critical_pass_rate"] == 1.0
    and (report_v1["metrics"]["recall_at_5"] - report_v2["metrics"]["recall_at_5"]) <= MAX_RECALL_DROP
    and (report_v1["metrics"]["mrr_at_5"] - report_v2["metrics"]["mrr_at_5"]) <= MAX_MRR_DROP
)
router.promote_candidate(gate_passed)

print("Antwort v2:", router.query(test_query))

Status vor CI-Gate: Aktiv = v1
Antwort v1: {'served_by': 'v1', 'top_hit': 'docs/release-notes.md#v2-3-postgres', 'score': 0.9072}
🟢 Gate bestanden: Schalte Produktions-Traffic unterbrechungsfrei auf v2 um!
Antwort v2: {'served_by': 'v2', 'top_hit': 'docs/release-notes.md#v2-3-postgres', 'score': 0.7312}
